# Day 3 — Chunking, Metadata Filters & Hybrid Search

---

So far we've embedded **short sentences**. Real documents are books, PDFs, and web pages — thousands of words each. Today we learn:

1. **Chunking** — splitting long docs into embed-able pieces
2. **Metadata filters** in more depth
3. **Hybrid search** — combining semantic search with old-school keyword search


## 1. Why we can't embed a whole document

Embedding models have a **max token limit** (usually 256–8192 tokens). More importantly, even if you *could* embed a 100-page PDF into one vector, that vector would be a hopelessly vague average — the meaning of "the whole book."

**Solution: chunking.** Split the document into small pieces (a paragraph, a few sentences), embed each piece separately, store them all. When someone searches, you find the most relevant *chunks*.

That's it. Everything else on this day is just "how to chunk well."


## 2. Chunking strategies — the three you need

### a) Fixed-size chunking
Chop every 500 characters (or every 200 tokens). Simple, fast, dumb.

**Problem:** it happily cuts a sentence in half.

### b) Recursive chunking
Try to split on paragraphs first. If a paragraph is too big, split on sentences. If a sentence is too big, split on words. **Respect natural boundaries.**

This is the standard choice. Every RAG framework (LangChain, LlamaIndex, etc.) uses it by default.

### c) Semantic chunking
Use an embedding model to detect where topics change and split there. Smartest, but slower and more complex. Skip until you're doing a serious production RAG.

**For this course: use recursive chunking. Always.**


## 3. Chunk size & overlap

Two knobs matter:

- **Chunk size** — usually **200–500 tokens** (~1–3 short paragraphs). Bigger = more context but vaguer meaning. Smaller = more precise but might lose context.
- **Overlap** — usually **10–20% of chunk size**. Overlap means each chunk repeats a bit of the previous one, so a sentence sitting on the boundary isn't lost.

Default recipe for freshers: **500 tokens, 50 token overlap.** Tune later.


## 4. Recursive chunking in practice


In [1]:
def recursive_chunk(text: str, chunk_size: int = 500, overlap: int = 50):
    """Split text into overlapping chunks, respecting paragraph & sentence boundaries."""
    # 1. Split on double newlines (paragraphs)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= chunk_size:
            current = (current + "\n\n" + para).strip()
        else:
            if current:
                chunks.append(current)
            # If the paragraph itself is too big, hard-split it
            while len(para) > chunk_size:
                chunks.append(para[:chunk_size])
                para = para[chunk_size - overlap:]
            current = para
    if current:
        chunks.append(current)

    # Add overlap between adjacent chunks
    with_overlap = []
    for i, c in enumerate(chunks):
        if i == 0:
            with_overlap.append(c)
        else:
            tail = chunks[i - 1][-overlap:]
            with_overlap.append(tail + c)
    return with_overlap


sample = """Python is a high-level programming language.
It was created by Guido van Rossum in 1991.

FastAPI is a modern web framework built on Python.
It uses type hints and Pydantic for validation.

Machine learning is a field of AI.
It focuses on models that learn from data.
"""

for i, c in enumerate(recursive_chunk(sample, chunk_size=120, overlap=20)):
    print(f"--- chunk {i} ({len(c)} chars) ---")
    print(c)
    print()


--- chunk 0 (88 chars) ---
Python is a high-level programming language.
It was created by Guido van Rossum in 1991.

--- chunk 1 (118 chars) ---
 van Rossum in 1991.FastAPI is a modern web framework built on Python.
It uses type hints and Pydantic for validation.

--- chunk 2 (97 chars) ---
ntic for validation.Machine learning is a field of AI.
It focuses on models that learn from data.



In a real project you'd use LangChain's `RecursiveCharacterTextSplitter` or LlamaIndex's `SentenceSplitter` — they handle edge cases better. But the idea is exactly what you just wrote.


## 5. Metadata: what to store per chunk

For every chunk, store:

- `source` — filename or URL
- `page` — page number (for PDFs)
- `chunk_index` — position within the document
- Anything else that helps filtering (`author`, `year`, `customer_id`, `department`)

**Why:** when you return search results, you want to say *"from page 12 of Contract_2024.pdf"* — not just show a wall of text. And you want to filter *"only search this customer's documents."*


In [5]:
import sys
import subprocess
import importlib.util

for pkg, import_name in [
    ("sentence-transformers", "sentence_transformers"),
    ("chromadb", "chromadb"),
    ("rank-bm25", "rank_bm25"),
]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            pkg,
            "--quiet",
            "--break-system-packages",
        ])

from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

# Use a persistent local database so the data survives restarts.
client = chromadb.PersistentClient(path="./chroma_data")
col = client.get_or_create_collection(name="knowledge_base")

# Simulate chunks from two source documents
chunks = [
    ("Python is a popular language.", "python_intro.md", 1),
    ("It has readable, elegant syntax.", "python_intro.md", 1),
    ("FastAPI is fast and modern.", "fastapi_guide.md", 3),
    ("It uses Pydantic for request validation.", "fastapi_guide.md", 3),
]

if col.count() == 0:
    col.add(
        documents=[c[0] for c in chunks],
        embeddings=model.encode([c[0] for c in chunks]).tolist(),
        metadatas=[{"source": c[1], "page": c[2]} for c in chunks],
        ids=[f"chunk_{i}" for i in range(len(chunks))],
    )
    print("Added chunks to the collection.")
else:
    print("Collection already has data; skipping add step.")

q = model.encode(["request validation"]).tolist()
r = col.query(query_embeddings=q, n_results=2, where={"source": "fastapi_guide.md"})
for doc, meta in zip(r["documents"][0], r["metadatas"][0]):
    print(f"  {meta['source']} p{meta['page']}: {doc}")


Collection already has data; skipping add step.
  fastapi_guide.md p3: It uses Pydantic for request validation.
  fastapi_guide.md p3: FastAPI is fast and modern.


## 6. Hybrid search — semantic + keyword

Semantic search is amazing for vague queries but weak for **exact matches**: product SKUs, error codes, proper nouns, part numbers. If a user searches for `"ERR-4041"`, a pure semantic search might not surface documents that mention it verbatim.

**Solution: hybrid search.** Do a keyword search **and** a semantic search, then combine the results.

- Keyword side: **BM25** (a battle-tested keyword-matching algorithm from the 1990s — used in Elasticsearch)
- Semantic side: your vector DB
- Combine: merge the two ranked lists so results that show up well in *both* rise to the top

**You don't need to know the BM25 math.** Libraries handle it.


In [ ]:
!pip install rank-bm25 --quiet

In [2]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util
import numpy as np

docs = [
    "Python is a popular programming language.",
    "Error code ERR-4041 means the file was not found.",
    "FastAPI is a modern Python web framework.",
    "The Eiffel Tower is in Paris.",
    "Machine learning trains models on data.",
]

# --- Keyword side: BM25 ---
tokenized = [d.lower().split() for d in docs]
bm25 = BM25Okapi(tokenized)

# --- Semantic side: embeddings ---
model = SentenceTransformer("all-MiniLM-L6-v2")
doc_vecs = model.encode(docs)

def hybrid_search(query: str, top_k: int = 3, alpha: float = 0.5):
    # keyword scores
    kw_scores = bm25.get_scores(query.lower().split())
    kw_scores = kw_scores / (kw_scores.max() or 1)          # normalize 0..1

    # semantic scores
    q_vec = model.encode(query)
    sem_scores = util.cos_sim(q_vec, doc_vecs).numpy()[0]   # already 0..1 ish

    # combine
    combined = alpha * kw_scores + (1 - alpha) * sem_scores
    top = np.argsort(-combined)[:top_k]
    return [(docs[i], combined[i]) for i in top]

for query in ["ERR-4041", "web framework in Python", "landmarks in France"]:
    print(f"\nQuery: {query!r}")
    for doc, s in hybrid_search(query):
        print(f"  {s:.3f}  {doc}")



Query: 'ERR-4041'
  0.886  Error code ERR-4041 means the file was not found.
  0.022  FastAPI is a modern Python web framework.
  0.016  Python is a popular programming language.

Query: 'web framework in Python'
  0.823  FastAPI is a modern Python web framework.
  0.412  The Eiffel Tower is in Paris.
  0.384  Python is a popular programming language.

Query: 'landmarks in France'
  0.747  The Eiffel Tower is in Paris.
  0.048  Machine learning trains models on data.
  0.022  Error code ERR-4041 means the file was not found.


**What just happened:**

- For `"ERR-4041"` the BM25 side dominates — an exact word match.
- For `"landmarks in France"` the semantic side dominates — the doc says "Eiffel Tower / Paris," not the query words.
- Combining both means you don't have to pick one — you get the best of both.

`alpha` is the blending knob: `alpha=1` = pure keyword, `alpha=0` = pure semantic, `alpha=0.5` = equal mix. **Start with 0.5 and adjust based on results.**

In production, tools like Weaviate and Elasticsearch do hybrid search out of the box.


## Recap

- Long documents → **chunk** them. Use **recursive** chunking with ~500 tokens + 50 overlap.
- Store **metadata** per chunk (source, page, author) so you can filter and cite.
- **Hybrid search** = keyword (BM25) + semantic (vectors). Best for queries with exact terms.
- **Next class:** we glue everything together into a Semantic Search Engine over real PDFs.
